# 🗄️ SQL Sales Analysis

> **Database:** SQLite (sales.db) with 3 tables — customers, products, orders  
> **Goal:** Answer real business questions using SQL, from basic to advanced.

## Tables
- `customers` — 200 customers with city and signup date
- `products` — 18 products across 6 categories
- `orders` — 2,000 orders with revenue, discount, returns

## Query Index
| # | Question | SQL Concept |
|---|----------|-------------|
| 1 | Overall summary | Basic aggregation |
| 2 | Revenue by category | GROUP BY + JOIN |
| 3 | Top 5 products | ORDER BY + LIMIT |
| 4 | Monthly trend | Date functions |
| 5 | Revenue by city | 3-table JOIN |
| 6 | Return rate | CASE / division |
| 7 | Top 10 customers | JOIN + GROUP BY |
| 8 | Running total | Window: SUM OVER |
| 9 | Product rank by category | Window: RANK OVER PARTITION |
| 10 | Month-over-month growth | Window: LAG |
| 11 | Above-average customers | Subquery in HAVING |
| 12 | Best month per category | CTE + Window combined |

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
COLORS = sns.color_palette('muted', 10)

conn = sqlite3.connect('data/sales.db')
print('Connected to sales.db')

## Query 1 — Overall Summary
**Concept:** Basic aggregation with COUNT, SUM, AVG

**Business question:** How much did we make in total?

In [ ]:
pd.read_sql("""
    SELECT
        COUNT(order_id)       AS total_orders,
        ROUND(SUM(revenue),2) AS total_revenue,
        ROUND(AVG(revenue),2) AS avg_order_value
    FROM orders
""", conn)

## Query 2 — Revenue by Category
**Concept:** GROUP BY + JOIN two tables

**Business question:** Which category makes the most money?

In [ ]:
q2 = pd.read_sql("""
    SELECT p.category,
           COUNT(o.order_id)       AS total_orders,
           ROUND(SUM(o.revenue),2) AS total_revenue
    FROM orders o
    JOIN products p ON o.product_id = p.product_id
    GROUP BY p.category
    ORDER BY total_revenue DESC
""", conn)

fig, ax = plt.subplots(figsize=(9,5))
ax.barh(q2['category'], q2['total_revenue'], color=COLORS[:len(q2)])
ax.set_xlabel('Revenue (₹)')
ax.set_title('Revenue by Category')
plt.tight_layout()
plt.show()
q2

## Query 3 — Top 5 Products
**Concept:** ORDER BY + LIMIT

**Business question:** What are our best-selling products?

In [ ]:
pd.read_sql("""
    SELECT p.product_name, p.category,
           COUNT(o.order_id)       AS times_ordered,
           ROUND(SUM(o.revenue),2) AS total_revenue
    FROM orders o
    JOIN products p ON o.product_id = p.product_id
    GROUP BY p.product_name
    ORDER BY total_revenue DESC
    LIMIT 5
""", conn)

## Query 4 — Monthly Revenue Trend
**Concept:** STRFTIME to extract month from date

**Business question:** Which months perform best?

In [ ]:
q4 = pd.read_sql("""
    SELECT STRFTIME('%Y-%m', order_date) AS month,
           COUNT(order_id)               AS total_orders,
           ROUND(SUM(revenue),2)         AS monthly_revenue
    FROM orders
    GROUP BY month
    ORDER BY month
""", conn)

fig, ax = plt.subplots(figsize=(12,4))
ax.plot(q4['month'], q4['monthly_revenue'], marker='o', color=COLORS[0], linewidth=2)
ax.fill_between(q4['month'], q4['monthly_revenue'], alpha=0.15, color=COLORS[0])
ax.set_title('Monthly Revenue Trend')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()
q4

## Query 8 — Running Total (Cumulative Revenue)
**Concept:** Window function — SUM() OVER (ORDER BY)

**Business question:** How is our revenue building up over the year?

> 💡 A **window function** does a calculation *across rows* without collapsing them into one. Here we're adding up revenue month by month.

In [ ]:
q8 = pd.read_sql("""
    WITH monthly AS (
        SELECT STRFTIME('%Y-%m', order_date) AS month,
               ROUND(SUM(revenue),2)         AS monthly_revenue
        FROM orders GROUP BY month
    )
    SELECT month, monthly_revenue,
           ROUND(SUM(monthly_revenue) OVER (ORDER BY month),2) AS running_total
    FROM monthly ORDER BY month
""", conn)

fig, ax = plt.subplots(figsize=(12,4))
ax.plot(q8['month'], q8['running_total'], marker='s', color=COLORS[3], linewidth=2)
ax.set_title('Cumulative Revenue — 2023')
ax.set_ylabel('Running Total (₹)')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()
q8

## Query 9 — Rank Products Within Category
**Concept:** RANK() OVER (PARTITION BY)

**Business question:** What is the #1 product in each category?

> 💡 **PARTITION BY** is like GROUP BY but for window functions. It resets the rank counter for each category.

In [ ]:
pd.read_sql("""
    WITH product_revenue AS (
        SELECT p.category, p.product_name,
               ROUND(SUM(o.revenue),2) AS total_revenue
        FROM orders o
        JOIN products p ON o.product_id = p.product_id
        GROUP BY p.category, p.product_name
    )
    SELECT category, product_name, total_revenue,
           RANK() OVER (PARTITION BY category ORDER BY total_revenue DESC) AS rank_in_category
    FROM product_revenue
    ORDER BY category, rank_in_category
""", conn)

## Query 10 — Month-over-Month Growth
**Concept:** LAG() window function

**Business question:** Did revenue go up or down compared to last month?

> 💡 **LAG()** lets you look at the *previous row's value*. Super useful for comparing periods.

In [ ]:
q10 = pd.read_sql("""
    WITH monthly AS (
        SELECT STRFTIME('%Y-%m', order_date) AS month,
               ROUND(SUM(revenue),2)         AS monthly_revenue
        FROM orders GROUP BY month
    )
    SELECT month, monthly_revenue,
           LAG(monthly_revenue) OVER (ORDER BY month) AS prev_month,
           ROUND((monthly_revenue - LAG(monthly_revenue) OVER (ORDER BY month))
                 * 100.0 / LAG(monthly_revenue) OVER (ORDER BY month), 1) AS growth_pct
    FROM monthly ORDER BY month
""", conn)

fig, ax = plt.subplots(figsize=(12,4))
colors = ['green' if x >= 0 else 'red' for x in q10['growth_pct'].fillna(0)]
ax.bar(q10['month'], q10['growth_pct'].fillna(0), color=colors)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Month-over-Month Revenue Growth (%)')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()
q10

## Key Business Insights 🔍

| # | Insight |
|---|---------|
| 1 | Electronics accounts for ~77% of all revenue |
| 2 | Laptop is the single highest revenue product |
| 3 | Hyderabad is the top revenue city |
| 4 | May and June are the strongest months |
| 5 | Books have the highest return rate (15.4%) |
| 6 | Revenue grew steadily to ₹4.65L by year end |